# 生成数据集:3 个环境 x 2 个脚本 = 6 个 case

环境:`cartpole` / `mountain_car` / `pendulum`

脚本:
- `make_decoder_dataset` -> 生成训练 Decoder(WM)用的 (state, image) 数据对 + StarV 对齐的初始 state 和真实闭环轨迹
- `sampling` -> 用指定 decoder（old / intensity / saliency）跑闭环验证轨迹（transition 数据生成已停用，见 sampling.py 注释）

**运行前**:Jupyter kernel 切换成 `starv_shared`(`/home/tealab_shared/starv/env/starv_shared`)。

本文件放在 `verifiable_wm/notebooks/` 下,第一个 cell 会自动定位仓库根目录并切换过去。

In [ ]:
import os
os.environ.setdefault("PYGLET_HEADLESS", "True")

import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "dynamic.py").exists():
    NOTEBOOK_DIR = NOTEBOOK_DIR.parent
assert (NOTEBOOK_DIR / "dynamic.py").exists(), (
    f"没找到仓库根目录（dynamic.py），请从 verifiable_wm/ 或 verifiable_wm/notebooks/ 启动 Jupyter"
)
os.chdir(NOTEBOOK_DIR)
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

print("repo root:", NOTEBOOK_DIR)

In [ ]:
from utils import load_config
import make_decoder_dataset as mdd
import sampling as sp

def run_make_decoder_dataset(env_name, starv_only=False):
    """starv_only=True: 只重新生成 StarV 对齐的那套产物（starv_states.npz + 从它出发的
    real_trajectories.npz），不重写 decoder 训练用的 decoder_states.npz/图片/权重。等价命令行：
    python make_decoder_dataset.py config/make_decoder_dataset/<env>.json --starv-only"""
    config_path = NOTEBOOK_DIR / "config" / "make_decoder_dataset" / f"{env_name}.json"
    config = load_config(config_path)
    print(f"[Config] {config_path}")
    output_dir = mdd.generate_dataset(config, starv_only=starv_only)
    print(f"[Done] decoder dataset saved to {output_dir}")
    return output_dir

def run_sampling(env_name, decoder_variant=None):
    """decoder_variant: 闭环轨迹用哪个 decoder 跑（old / intensity / saliency）。
    None 时用 config 里 decoder.variant 的值；输出文件为 dwm_trajectories_<variant>.npz。"""
    config_path = NOTEBOOK_DIR / "config" / "sampling" / f"{env_name}.json"
    config = load_config(config_path)
    if decoder_variant is not None:
        config["decoder"]["variant"] = decoder_variant
    print(f"[Config] {config_path}")
    output_dir = sp.generate_dataset(config)
    print(f"[Done] dwm trajectories saved to {output_dir}")
    return output_dir

## Case 1: CartPole - make_decoder_dataset

完整生成，写入 `datasets/cartpole/data/dataset_v1/` 下三个文件：

- `decoder_states.npz`：decoder 训练数据（`{split}_states` + 渲染的 `{split}_images`）
- `starv_states.npz`：从 StarV grid 采样的完整初始 state
- `real_trajectories.npz`：从这些 state 出发、用真实 renderer 跑的闭环轨迹

生成是确定性的（范围和 seed 都在 config 里、config 进 git），怀疑磁盘数据过期时直接重跑本 cell 即可。

In [ ]:
run_make_decoder_dataset("cartpole")

## Case 1b: CartPole - starv-only

只生成两个文件，不动 decoder 训练数据：

- `starv_states.npz`：从 StarV grid 采样的完整初始 state
- `real_trajectories.npz`：从这些 state 出发、用真实 renderer 跑的闭环轨迹

In [ ]:
run_make_decoder_dataset("cartpole", starv_only=True)

## Case 2: CartPole - sampling

读取 `starv_states.npz` 作为初始 state（所以必须先跑 Case 1 或 1b），生成：

- `dwm_trajectories_<variant>.npz`：用指定 decoder 替代 renderer 的闭环轨迹，起点和 real 完全相同

（`transition_dataset.npz` 的生成已停用——它是 learned dynamics 用的 (s,a,s') 数据，当前没有下游消费者，
需要时在 `sampling.py` 里取消注释即可恢复。）

In [ ]:
# 闭环轨迹用哪个 decoder 跑：三选一，注释掉另外两个
# 输出互不覆盖，分别存为 dwm_trajectories_saliency / _intensity / _old.npz

run_sampling("cartpole", decoder_variant="saliency")     # 我们的方法（主线，消融时反复跑这个）
# run_sampling("cartpole", decoder_variant="intensity")  # 论文 baseline（跑一次即可）
# run_sampling("cartpole", decoder_variant="old")        # 一代老 decoder（可选的一次性参照）

## Case 3: MountainCar - make_decoder_dataset

产物同 Case 1，目录换成 `datasets/mountain_car/data/dataset_v1/`。

In [ ]:
run_make_decoder_dataset("mountain_car")

## Case 3b: MountainCar - starv-only

产物同 Case 1b（`starv_states.npz` + `real_trajectories.npz`），目录换成 `datasets/mountain_car/data/dataset_v1/`。

In [ ]:
run_make_decoder_dataset("mountain_car", starv_only=True)

## Case 4: MountainCar - sampling

产物同 Case 2，目录换成 `datasets/mountain_car/data/dataset_v1/`。

In [ ]:
# MC 冻结中：decoder 只有一份老权重（config 里 weights 是单个路径，不分 variant），
# 输出固定为 dwm_trajectories_old.npz
run_sampling("mountain_car")

## Case 5: Pendulum - make_decoder_dataset

产物同 Case 1，目录换成 `datasets/pendulum/data/dataset_v1/`。

In [ ]:
run_make_decoder_dataset("pendulum")

## Case 5b: Pendulum - starv-only

产物同 Case 1b（`starv_states.npz` + `real_trajectories.npz`），目录换成 `datasets/pendulum/data/dataset_v1/`。

In [ ]:
run_make_decoder_dataset("pendulum", starv_only=True)

## Case 6: Pendulum - sampling

产物同 Case 2，目录换成 `datasets/pendulum/data/dataset_v1/`。

In [ ]:
# 闭环轨迹用哪个 decoder 跑：三选一，注释掉另外两个
# 输出互不覆盖，分别存为 dwm_trajectories_saliency / _intensity / _old.npz

run_sampling("pendulum", decoder_variant="saliency")     # 我们的方法（主线，消融时反复跑这个）
# run_sampling("pendulum", decoder_variant="intensity")  # 论文 baseline（跑一次即可）
# run_sampling("pendulum", decoder_variant="old")        # 一代老 decoder（可选的一次性参照）

## 汇总:检查所有生成的文件

In [ ]:
for env_name in ("cartpole", "mountain_car", "pendulum"):
    data_dir = NOTEBOOK_DIR / "datasets" / env_name / "data" / "dataset_v1"
    print(f"--- {env_name} ({data_dir}) ---")
    if data_dir.exists():
        for f in sorted(data_dir.iterdir()):
            print(f"  {f.name}: {f.stat().st_size} bytes")
    else:
        print("  (尚未生成)")